In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

import autosklearn.classification

RANDOM_STATE = 42
DATA_PATH = "data/dating_app_behavior_dataset.csv"

df = pd.read_csv(DATA_PATH)

def build_dataset(include_usage_time=True):
    target_col = "app_usage_time_label"

    # Multi-hot encode interest tags
    interest_dummies = df["interest_tags"].fillna("").str.get_dummies(sep=", ")
    interest_dummies = interest_dummies.add_prefix("interest_")

    model_df = pd.concat([df.drop(columns=["interest_tags"]), interest_dummies], axis=1)

    # match_outcome is kept for analysis only, not prediction
    drop_cols = [target_col, "match_outcome"]

    # Strict mode removes the direct target-defining field
    if not include_usage_time:
        drop_cols.append("app_usage_time_min")

    X = model_df.drop(columns=drop_cols)
    y = model_df[target_col]

    # One-hot encode remaining categorical features.
    # auto-sklearn 0.15 can fail on bool columns, so force dummy columns to integers.
    X = pd.get_dummies(X, drop_first=False, dtype=np.int8)
    bool_cols = X.select_dtypes(include=["bool"]).columns
    X[bool_cols] = X[bool_cols].astype(np.int8)

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    return train_test_split(
        X,
        y_encoded,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_encoded
    ), label_encoder

def run_autosklearn(include_usage_time=True, time_limit=600):
    (X_train, X_test, y_train, y_test), label_encoder = build_dataset(include_usage_time)

    automl = autosklearn.classification.AutoSklearnClassifier(
        time_left_for_this_task=time_limit,
        per_run_time_limit=60,
        ensemble_kwargs={"ensemble_size": 50},
        seed=RANDOM_STATE,
        n_jobs=-1,
        metric=autosklearn.metrics.f1_macro
    )

    automl.fit(X_train, y_train)
    y_pred = automl.predict(X_test)

    print("Mode:", "High-accuracy / non-strict" if include_usage_time else "Strict no-leakage")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_test, y_pred, average="weighted"), 4))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    print("\nLeaderboard:")
    print(automl.leaderboard())

    return automl

# Strict version: fair comparison with your strict notebook
autosklearn_strict = run_autosklearn(include_usage_time=False, time_limit=600)

# High-accuracy version: includes app_usage_time_min
autosklearn_high_accuracy = run_autosklearn(include_usage_time=True, time_limit=600)

/home/sprball/miniforge3/envs/autosklearn39/lib/python3.9/site-packages/sklearn/utils/fixes.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version  # type: ignore
/home/sprball/miniforge3/envs/autosklearn39/lib/python3.9/site-packages/sklearn/utils/fixes.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version  # type: ignore


[WARNING] [2026-06-04 17:44:02,895:Client-EnsembleBuilder] No runs were available to build an ensemble from
Mode: Strict no-leakage
Accuracy: 0.2413
Macro F1: 0.1408
Weighted F1: 0.2459

Classification report:
              precision    recall  f1-score   support

    Addicted       0.21      0.25      0.23      1982
      Barely       0.01      0.01      0.01       177
Extreme User       0.40      0.32      0.36      4028
        High       0.20      0.23      0.21      1982
         Low       0.07      0.06      0.06       507
    Moderate       0.10      0.13      0.11      1000
    Very Low       0.01      0.00      0.00       324

    accuracy                           0.24     10000
   macro avg       0.14      0.14      0.14     10000
weighted avg       0.26      0.24      0.25     10000


Leaderboard:
          rank  ensemble_weight               type      cost   duration
model_id                                                               
120          1             0.06    